# Cell Probe Prediction And Probe-Weight Viewers

Interactive inspection of cell-level candidate probe predictions on a Sudoku grid, plus cosine similarity maps between probe weight vectors for a selected digit.


In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from sudoku.probes.session import ProbeSession
from sudoku.probes.modes import cell_candidates
from sudoku.probes.activations import build_grid_at_step
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, END_CLUES_TOKEN, PAD_TOKEN_BT

In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
LAYER = 4     # layer to probe (0-indexed)
N_CELLS = 81    # probe all cells
SEED = 42

## Load session

In [ ]:
session = ProbeSession(CACHE_PATH)

## Classify puzzles: simple vs hard

Simple = no PUSH/POP tokens anywhere in the sequence (pure constraint propagation).  
Hard = at least one PUSH token (backtracking was used).

In [ ]:
is_simple = np.array([
    not any(t == PUSH_TOKEN for t in seq)
    for seq in session.sequences
])

n_simple = is_simple.sum()
n_hard = (~is_simple).sum()
print(f"Simple puzzles (no backtracking): {n_simple:,}")
print(f"Hard puzzles (with backtracking): {n_hard:,}")

# Sequence length distribution check
seq_lens = np.array([len(s) for s in session.sequences])
print(f"\nSimple seq len: min={seq_lens[is_simple].min()}, max={seq_lens[is_simple].max()}, mean={seq_lens[is_simple].mean():.1f}")
print(f"Hard   seq len: min={seq_lens[~is_simple].min()}, max={seq_lens[~is_simple].max()}, mean={seq_lens[~is_simple].mean():.1f}")

## Build train/test split at step 0

- **Train**: all puzzles at step 0 (80% split)
- **Test**: the remaining 20%, further divided into simple/hard subsets

In [ ]:
idx0 = session.index.at_step(1).first_per_puzzle()
train_mask, test_mask = session.split(idx0, test_size=0.2, seed=SEED)

train_idx = idx0[train_mask]
test_idx  = idx0[test_mask]

# Identify which test puzzles are simple vs hard
test_puzzle_ids = test_idx.puzzle_idx
test_is_simple = is_simple[test_puzzle_ids]

print(f"Train: {len(train_idx):,}  |  Test: {len(test_idx):,}")
print(f"Test simple: {test_is_simple.sum():,}  |  Test hard: {(~test_is_simple).sum():,}")

## Helper functions

In [ ]:
from scripts.analysis.probe_experiments import (
    build_candidate_targets,
    fit_multilabel_probe as fit_cell_probe,
    eval_multilabel_probe as eval_cell_probe,
)


## Train probes at step 0

One multi-label LR probe per cell (81 probes total), trained on the training split at step 0.

In [ ]:
# Activations at step 0 for train and test splits
X_train_all = session.acts(train_idx, layer=LAYER)  # (n_train, d_model)
X_test_all  = session.acts(test_idx, layer=LAYER)   # (n_test,  d_model)

grids_train = session.grids(train_idx)
grids_test  = session.grids(test_idx)

print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

In [ ]:
# Train one probe per cell — filter to cells that have empty instances in train
cell_probes = {}  # cell_idx -> (clf, train_empty_mask)

for cell_idx in tqdm(range(N_CELLS), desc="Training probes"):
    targets_train, is_empty_train = build_candidate_targets(grids_train, cell_idx)
    if is_empty_train.sum() < 10:
        continue  # skip cells with almost no empty instances in train
    clf = fit_cell_probe(X_train_all[is_empty_train], targets_train[is_empty_train])
    cell_probes[cell_idx] = clf

print(f"Trained probes for {len(cell_probes)} cells")

In [ ]:
# Pre-identify test puzzle IDs for quick lookup
test_puzzle_id_set = set(test_puzzle_ids.tolist())
simple_test_puzzle_ids = set(test_puzzle_ids[test_is_simple].tolist())
hard_test_puzzle_ids   = set(test_puzzle_ids[~test_is_simple].tolist())


## Cell Candidate Viewer
Shows a 3x3 probability mini-grid inside each empty cell. Bold digits are true candidates.


In [ ]:
def get_puzzle_at_step(session, puzzle_id: int, step: int):
    idx = session.index.at_step(step)

    idx = idx[idx.puzzle_idx == puzzle_id]
    if len(idx) == 0:
        raise ValueError(f"Puzzle {puzzle_id} never reaches step {step} "
                         f"(seq len={len(session.sequences[puzzle_id])})")
    return idx[:1]

def get_puzzle_at_nempty(session, puzzle_id: int, n_empty: int):
    """Return a single-sample ActivationIndex for puzzle_id at n_empty empty cells.

    Selects the *first* position in the trace where exactly n_empty cells remain
    unfilled (forward pass; ignores later revisits due to backtracking).

    Raises ValueError if the puzzle never reaches that fill count.
    """
    idx = session.index.where_filled(81 - n_empty)
    idx = idx[idx.puzzle_idx == puzzle_id]
    if len(idx) == 0:
        raise ValueError(
            f"Puzzle {puzzle_id} never reaches n_empty={n_empty} "
            f"(seq len={len(session.sequences[puzzle_id])})"
        )
    return idx[:1]   # first occurrence = forward pass


def show_probe_state(session, cell_probes,
                     idx_entry=None,
                     puzzle_id: int | None = None,
                     n_empty: int | None = None,
                     layer=LAYER):
    """Visualise probe candidate predictions for one board snapshot.

    Supply either:
      idx_entry                — a single-sample ActivationIndex (use slice [:1])
      puzzle_id  +  n_empty   — convenience; calls get_puzzle_at_nempty internally

    Each empty cell shows a 3×3 sub-grid of per-digit predicted probabilities
    (green=high, red=low).  Bold digit = true candidate per current board state.
    Filled cells are shown in grey with their digit centred.
    """
    if idx_entry is None:
        if puzzle_id is None or n_empty is None:
            raise ValueError("Supply either idx_entry or both puzzle_id and n_empty")
        idx_entry = get_puzzle_at_nempty(session, puzzle_id, n_empty)

    assert len(idx_entry) == 1, "Pass a single-sample index (use idx[:1] not idx[0])"

    X        = session.acts(idx_entry, layer=layer)  # (1, d_model)
    grid_str = session.grids(idx_entry)[0]            # 81-char board-state string

    fig, ax = plt.subplots(figsize=(9.5, 9.5))
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 9)
    ax.set_aspect('equal')
    ax.axis('off')

    cmap = plt.cm.RdYlGn

    for cell_idx in range(81):
        r, c = divmod(cell_idx, 9)
        y0   = 8 - r   # bottom-left y of this cell in plot coords
        ch   = grid_str[cell_idx]

        if ch != '0':
            ax.add_patch(plt.Rectangle((c, y0), 1, 1, color='#cccccc', zorder=1))
            ax.text(c + 0.5, y0 + 0.5, ch,
                    ha='center', va='center', fontsize=22, fontweight='bold', zorder=2)

        elif cell_idx in cell_probes:
            clf        = cell_probes[cell_idx]
            proba_list = clf.predict_proba(X)
            probas     = np.array([p[0, 1] for p in proba_list])  # (9,)
            true_cands = cell_candidates(grid_str, cell_idx)       # list[int] length 9

            for d in range(9):
                dr, dc = divmod(d, 3)
                sx = c + dc / 3
                sy = y0 + (2 - dr) / 3   # dr=0 → top row → largest y
                ax.add_patch(plt.Rectangle((sx, sy), 1/3, 1/3,
                                           color=cmap(float(probas[d])), zorder=1))
                ax.text(sx + 1/6, sy + 1/6, str(d + 1),
                        ha='center', va='center', fontsize=7, zorder=2,
                        fontweight='bold' if true_cands[d] else 'normal',
                        color='black'     if true_cands[d] else '#666666')
        else:
            ax.text(c + 0.5, y0 + 0.5, '?',
                    ha='center', va='center', fontsize=14, color='grey')

    for i in range(10):
        lw = 2.5 if i % 3 == 0 else 0.5
        ax.axhline(i, color='black', linewidth=lw, zorder=3)
        ax.axvline(i, color='black', linewidth=lw, zorder=3)

    n_empty_actual = grid_str.count('0')
    ax.set_title(
        f"Puzzle {int(idx_entry.puzzle_idx[0])}  |  "
        f"seq_pos={int(idx_entry.seq_pos[0])}  |  n_empty={n_empty_actual} | token={idx_entry.tokens[0]}\n"
        "Background: P(candidate) — green=1, red=0  |  Bold digit = true candidate",
        fontsize=11, pad=12,
    )
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.02, label='P(candidate)')
    plt.tight_layout()
    plt.savefig('backtrack.svg')
    plt.show()

In [ ]:
import ipywidgets as widgets

@widgets.interact(
    puzzle_id=widgets.Dropdown(
        options=sorted(hard_test_puzzle_ids),
        description="Puzzle:",
    ),
    n_empty=widgets.IntSlider(
        value=40, min=0, max=100, step=1,
        description="step:",
        continuous_update=False,   # only redraw on release, not while dragging
    ),
)
def _show_interactive(puzzle_id, n_empty):
    try:
        show_probe_state(session, cell_probes, puzzle_id=puzzle_id, n_empty=n_empty)
    except ValueError as e:
        print(e)

## Probe Weight Cosine Viewer
For one cell and digit, compare its logistic-regression probe vector with the same digit probe at every other cell.


In [ ]:
plt.rcParams['font.family'] = ['Avenir']

def get_probe_weight(cell_probes, cell_idx: int, digit: int) -> np.ndarray:
    """Return the weight vector (d_model,) for the binary LR probe at (cell, digit).
    digit is 1-indexed (1–9).
    """
    return cell_probes[cell_idx].estimators_[digit - 1].coef_[0]


def plot_probe_cosine_similarity(cell_probes, cell_idx: int, digit: int):
    """Plot cosine similarity of the probe w-vector for (cell_idx, digit)
    against every other cell's probe w-vector for the same digit.

    The reference cell is outlined in yellow.  Cells without a trained probe
    are left white (NaN).
    """
    if cell_idx not in cell_probes:
        raise ValueError(f"No probe trained for cell {cell_idx}")

    w_ref  = get_probe_weight(cell_probes, cell_idx, digit)
    norm_ref = np.linalg.norm(w_ref)

    sim_grid = np.full(81, np.nan)
    for c, _ in cell_probes.items():
        w = get_probe_weight(cell_probes, c, digit)
        sim_grid[c] = np.dot(w_ref, w) / (norm_ref * np.linalg.norm(w))

    sim_matrix = sim_grid.reshape(9, 9)

    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)

    for r in range(9):
        for c in range(9):
            v = sim_matrix[r, c]
            if not np.isnan(v):
                ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(v) > 0.65 else "black")

    # Highlight reference cell
    ref_r, ref_c = divmod(cell_idx, 9)
    ax.add_patch(plt.Rectangle(
        (ref_c - 0.5, ref_r - 0.5), 1, 1,
        fill=False, edgecolor="yellow", linewidth=3, zorder=3,
    ))

    # 3×3 block boundaries
    for pos in [2.5, 5.5]:
        ax.axhline(pos, color="black", linewidth=2)
        ax.axvline(pos, color="black", linewidth=2)

    ax.set_xticks(range(9))
    ax.set_xticklabels(range(1, 10))
    ax.set_yticks(range(9))
    ax.set_yticklabels(range(1, 10))
    # ax.set_title(
    #     f"Probe cosine similarity — digit {digit},  "
    #     f"reference cell {cell_idx} (r{ref_r} c{ref_c})",
    #     fontsize=11,
    # )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="cosine similarity")
    plt.tight_layout()
    plt.savefig("probe_cosine_sim.png", dpi=200)
    plt.show()


# --- Interactive widget ---
@widgets.interact(
    cell_idx=widgets.IntSlider(value=0, min=0, max=80, step=1,
                               description="Cell:", continuous_update=False),
    digit=widgets.IntSlider(value=7, min=1, max=9, step=1,
                            description="Digit:", continuous_update=False),
)
def _show_cosine(cell_idx, digit):
    if cell_idx not in cell_probes:
        print(f"No probe for cell {cell_idx}")
        return
    plot_probe_cosine_similarity(cell_probes, cell_idx, digit)